# Session 4.1 -- Observability and Debugging

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Instrument** a RAG pipeline with structured logging that captures query, embedding, retrieval, and generation stages
2. **Measure** key pipeline metrics: latency per stage, token usage, similarity scores, source attribution
3. **Diagnose** retrieval failures using a systematic debugging workflow (the "retrieval autopsy")
4. **Build** a diagnostic view that surfaces pipeline health indicators

## What You Are Working With

- `src/s5_observability/logger.py` -- `PipelineLogger`, `StageTimer`, and `load_logs()` (provided complete)
- `src/s4_retrieval/naive.py` -- Naive RAG pipeline from Session 3.1 (provided complete)
- `src/s4_retrieval/filtered.py` -- Metadata-filtered RAG from Session 3.2 (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions (provided complete)

You **import and use** the pre-built modules. The observability module is provided complete -- you explore it, wire it into the pipeline, and use the data it produces to diagnose issues.

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import json
import os
import time
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

# Observability module (provided complete)
from src.s5_observability.logger import PipelineLogger, StageTimer, load_logs

# Retrieval modules from Sessions 3.1 and 3.2
from src.s4_retrieval.naive import naive_rag, naive_retrieve
from src.s4_retrieval.filtered import filtered_rag

# Embedding module from Session 2.1
from src.s2_embeddings.embed import embed_texts

print("All imports loaded successfully.")

In [ ]:
# Create the logs directory for this session
LOG_DIR = Path("../logs")
LOG_DIR.mkdir(exist_ok=True)

# Clear any previous session logs so this notebook is idempotent
log_file = LOG_DIR / "pipeline.jsonl"
if log_file.exists():
    log_file.unlink()
    print("Previous log file cleared.")

print(f"Log directory: {LOG_DIR.resolve()}")
print("Ready to start logging.")

---

## 2. Why Observability?

Your pipeline works. You can ask a question, retrieve chunks, and generate an answer.

But **does it work well?** How would you know if retrieval quality degraded? If latency doubled? If the model started hallucinating because the wrong chunks were retrieved?

Right now, your pipeline is a **black box**. A question goes in, an answer comes out. You have no visibility into what happens between those two points.

```
Question  -->  [  ???  ]  -->  Answer
```

Today we crack it open:

```
Question  -->  [Embed]  -->  [Retrieve]  -->  [Generate]  -->  Answer
                 |              |                 |
                 v              v                 v
              Latency        Scores           Tokens
              Model          Sources          Latency
                             Count            Model
                             Latency          Stop reason
```

By the end of this notebook, every query will leave a **structured trace** that you can inspect, aggregate, and use to diagnose problems.

> *"You cannot debug what you cannot see. In deterministic systems, you can reason about what happened. In non-deterministic systems, you must log what happened."*

---

## 3. Exploring StageTimer

`StageTimer` is a Python context manager that captures how long an operation takes, in milliseconds. You wrap any piece of work in a `with StageTimer() as t:` block, and after the block completes, `t.elapsed_ms` gives you the wall-clock time.

This is the building block for timing every pipeline stage.

In [ ]:
# Basic StageTimer demo -- time a sleep operation
with StageTimer() as t:
    time.sleep(0.1)  # 100ms

print(f"Elapsed: {t.elapsed_ms}ms")
print(f"Type: {type(t.elapsed_ms).__name__}")
print()
print("StageTimer captures wall-clock time in integer milliseconds.")
print("It uses time.perf_counter() under the hood for precision.")

In [ ]:
# Time a real operation: embedding a question
question = "What is the vacation policy at Northbrook Partners?"

with StageTimer() as t:
    embedding = embed_texts([question])[0]

print(f"Embedding '{question}'")
print(f"Elapsed: {t.elapsed_ms}ms")
print(f"Embedding dimensions: {len(embedding)}")
print()
print("Every API call has latency. StageTimer makes it visible.")

In [ ]:
# Time multiple operations and compare
operations = {
    "Short text embed": lambda: embed_texts(["hello"]),
    "Medium text embed": lambda: embed_texts(["What is the company policy on remote work and flexible scheduling?"]),
    "Batch embed (3 texts)": lambda: embed_texts([
        "First question about policies",
        "Second question about budgets",
        "Third question about onboarding",
    ]),
}

print(f"{'Operation':<30s}  {'Latency (ms)':>12s}")
print("-" * 45)

for name, op in operations.items():
    with StageTimer() as t:
        op()
    print(f"{name:<30s}  {t.elapsed_ms:>12d}")

print()
print("Notice: batching is more efficient than separate calls.")
print("This is exactly the kind of insight observability provides.")

---

## 4. Exploring PipelineLogger

`PipelineLogger` is the structured logging class for the RAG pipeline. It collects timing and metadata from every stage -- embed, retrieve, generate -- and produces a single JSON log entry when the query is done.

Let's walk through its API step by step.

In [ ]:
# Create a PipelineLogger instance
logger = PipelineLogger("What is the vacation policy?")

# Inspect the initial state
print("=== Initial Logger State ===")
print(json.dumps(logger.to_dict(), indent=2))
print()
print(f"query_id: {logger.query_id}  (UUID-based, unique per query)")
print(f"query:    {logger.query}")
print(f"stages:   {logger.stages}  (empty -- nothing logged yet)")

In [ ]:
# Log the embed stage manually
logger.log_embed(latency_ms=120, model="voyage-3-lite", input_chars=35)

print("After log_embed():")
print(json.dumps(logger.to_dict(), indent=2))

In [ ]:
# Log the retrieve stage manually
logger.log_retrieve(
    latency_ms=18,
    n_results=5,
    scores=[0.78, 0.65, 0.54, 0.43, 0.38],
    sources=["hr_policy.md", "hr_policy.md", "handbook.md", "memo.md", "handbook.md"],
    filter_applied=None,
)

print("After log_retrieve():")
print(json.dumps(logger.to_dict(), indent=2))

# Notice the derived metrics
retrieve = logger.to_dict()["stages"]["retrieve"]
print(f"\nDerived metrics:")
print(f"  top_score:      {retrieve['top_score']}")
print(f"  low_score:      {retrieve['low_score']}")
print(f"  score_spread:   {retrieve['score_spread']}")
print(f"  unique_sources: {retrieve['unique_sources']}")

In [ ]:
# Log the generate stage manually
logger.log_generate(
    latency_ms=1650,
    model="claude-sonnet-4-5",
    input_tokens=2400,
    output_tokens=280,
    stop_reason="end_turn",
)

print("After log_generate():")
print(json.dumps(logger.to_dict(), indent=2))

In [ ]:
# Finalize -- calculates totals and writes to JSONL file
result = logger.finalize(log_dir=str(LOG_DIR))

print("=== Finalized Log Entry ===")
print(json.dumps(result, indent=2))
print()
print(f"total_latency_ms: {result['total_latency_ms']}ms")
print(f"  embed:    {result['stages']['embed']['latency_ms']}ms")
print(f"  retrieve: {result['stages']['retrieve']['latency_ms']}ms")
print(f"  generate: {result['stages']['generate']['latency_ms']}ms")
print()
print(f"Written to: {LOG_DIR / 'pipeline.jsonl'}")

In [ ]:
# Verify the JSONL file was written
with open(LOG_DIR / "pipeline.jsonl") as f:
    line = f.readline().strip()
    written = json.loads(line)

print("Read back from JSONL file:")
print(f"  query:    {written['query']}")
print(f"  query_id: {written['query_id']}")
print(f"  total_ms: {written['total_latency_ms']}")
print()
print("JSONL = one JSON object per line. Appendable, queryable, portable.")
print("This is observability. Every query leaves a trace.")

---

## 5. Wiring It Into the Pipeline

Now we connect `PipelineLogger` and `StageTimer` to the actual RAG pipeline. Instead of logging fake data, we will run a real query through embed, retrieve, and generate -- and capture structured logs at every stage.

The integration pattern:
1. Create a `PipelineLogger` for the query
2. Wrap each stage with `StageTimer`
3. Call the appropriate `log_*` method after each stage
4. Call `finalize()` to write the log entry

In [ ]:
def logged_rag_query(question: str, use_filter: dict | None = None, top_k: int = 5) -> dict:
    """Run a RAG query with full observability logging.

    Wraps the existing pipeline modules with StageTimer and PipelineLogger
    to capture latency, scores, sources, and token usage at every stage.

    Args:
        question: The user's question.
        use_filter: Optional ChromaDB where clause for filtered retrieval.
        top_k: Number of chunks to retrieve.

    Returns:
        A dict with 'answer', 'sources', and 'log' keys.
    """
    logger = PipelineLogger(question)

    # --- Embed stage ---
    with StageTimer() as t:
        query_embedding = embed_texts([question])[0]
    logger.log_embed(
        latency_ms=t.elapsed_ms,
        model="voyage-3-lite",
        input_chars=len(question),
    )

    # --- Retrieve stage ---
    from src.s3_ingestion.store import get_collection
    collection = get_collection()

    with StageTimer() as t:
        if use_filter:
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                where=use_filter,
                include=["documents", "metadatas", "distances"],
            )
        else:
            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                include=["documents", "metadatas", "distances"],
            )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]
    scores = [round(1 - d, 4) for d in distances]  # distance to similarity
    sources_list = [m.get("source", "unknown") for m in metadatas]

    logger.log_retrieve(
        latency_ms=t.elapsed_ms,
        n_results=len(documents),
        scores=scores,
        sources=sources_list,
        filter_applied=use_filter,
    )

    # --- Generate stage ---
    # Build context from retrieved chunks
    context_blocks = []
    for doc, meta, score in zip(documents, metadatas, scores):
        source_name = meta.get("source", "Unknown")
        context_blocks.append(f"[Source: {source_name}, Score: {score:.3f}]\n{doc}")

    context_section = "\n\n---\n\n".join(context_blocks)

    system_prompt = (
        "You are a Q&A assistant for Northbrook Partners. "
        "Answer using ONLY the provided context. "
        "Cite your sources by name. "
        "If the context is insufficient, say: "
        "'I don't have enough information to answer that question.'"
    )
    user_message = f"Context:\n\n{context_section}\n\n---\n\nQuestion: {question}"

    from src.s0_generation.generate import call_claude_with_usage

    with StageTimer() as t:
        gen_result = call_claude_with_usage(
            prompt=user_message,
            system_prompt=system_prompt,
            temperature=0.0,
        )

    logger.log_generate(
        latency_ms=t.elapsed_ms,
        model=gen_result.get("model", "claude-sonnet-4-5"),
        input_tokens=gen_result["input_tokens"],
        output_tokens=gen_result["output_tokens"],
        stop_reason=gen_result.get("stop_reason", "end_turn"),
    )

    # --- Finalize ---
    log_entry = logger.finalize(log_dir=str(LOG_DIR))

    return {
        "answer": gen_result["text"],
        "sources": [
            {"text": doc, "metadata": meta, "score": score}
            for doc, meta, score in zip(documents, metadatas, scores)
        ],
        "log": log_entry,
    }


print("logged_rag_query() defined. Ready to run queries with observability.")

In [ ]:
# Clear the log file before our first real logged query
log_file = LOG_DIR / "pipeline.jsonl"
if log_file.exists():
    log_file.unlink()

# Run a single query with full logging
result = logged_rag_query("What is the vacation policy at Northbrook Partners?")

print("=== Answer ===")
print(result["answer"][:500])
print()
print("=== Log Entry ===")
print(json.dumps(result["log"], indent=2))

In [ ]:
# Break down the log entry
log = result["log"]

print("=== Pipeline Trace ===")
print(f"Query:    {log['query']}")
print(f"Query ID: {log['query_id']}")
print()

# Stage-by-stage breakdown
embed = log["stages"]["embed"]
retrieve = log["stages"]["retrieve"]
generate = log["stages"]["generate"]

print(f"{'Stage':<12s}  {'Latency':>10s}  {'Key Metric':>30s}")
print("-" * 55)
print(f"{'Embed':<12s}  {embed['latency_ms']:>8d}ms  {'model: ' + embed['model']:>30s}")
print(f"{'Retrieve':<12s}  {retrieve['latency_ms']:>8d}ms  {'top_score: ' + f"{retrieve['top_score']:.4f}":>30s}")
print(f"{'Generate':<12s}  {generate['latency_ms']:>8d}ms  {'tokens: ' + str(generate['input_tokens']) + ' in / ' + str(generate['output_tokens']) + ' out':>30s}")
print("-" * 55)
print(f"{'TOTAL':<12s}  {log['total_latency_ms']:>8d}ms")
print()
print(f"Sources: {retrieve['sources']}")
print(f"Unique sources: {retrieve['unique_sources']}")
print(f"Score spread: {retrieve['score_spread']}")

Every query now leaves a structured trace. You can see exactly how long each stage took, what the retrieval scores looked like, and how many tokens were consumed. No guessing.

---

## 6. Running Multiple Queries

One query is a data point. Ten queries show a pattern. Let's run a diverse set of questions through the logged pipeline and build up a log file we can analyze.

In [ ]:
# Clear the log file so we start fresh for analysis
log_file = LOG_DIR / "pipeline.jsonl"
if log_file.exists():
    log_file.unlink()
    print("Log file cleared.")

# 10 diverse queries -- a mix of topics, specificity, and expected difficulty
test_queries = [
    "What is the vacation policy at Northbrook Partners?",
    "How does the remote work policy work?",
    "What is the professional development budget?",
    "Who is on the leadership team?",
    "What was discussed at the most recent board meeting?",
    "What are the security protocols for the office?",
    "How does the Navigator onboarding program work?",
    "What is the dress code policy?",
    "What are the reimbursement limits for client entertainment?",
    "How does the performance review process work?",
]

print(f"Running {len(test_queries)} queries through the logged pipeline...")
print()

all_results = []

for i, query in enumerate(test_queries):
    result = logged_rag_query(query)
    log = result["log"]
    top_score = log["stages"]["retrieve"]["top_score"]
    total_ms = log["total_latency_ms"]

    # Flag potential issues
    flag = ""
    if top_score < 0.5:
        flag = "  << LOW SCORE"
    if total_ms > 5000:
        flag += "  << SLOW"

    print(f"  [{i+1:>2d}] {total_ms:>5d}ms  top={top_score:.4f}  {query[:55]}...{flag}")
    all_results.append(result)

print(f"\nAll {len(test_queries)} queries logged to {log_file}")

In [ ]:
# Verify the JSONL file has all entries
logs = load_logs(log_dir=str(LOG_DIR))

print(f"Log file contains {len(logs)} entries.")
print()
print(f"{'#':>3s}  {'Query ID':<10s}  {'Total ms':>10s}  {'Top Score':>10s}  {'Query':<50s}")
print("-" * 90)

for i, log in enumerate(logs):
    print(
        f"{i+1:>3d}  "
        f"{log['query_id']:<10s}  "
        f"{log['total_latency_ms']:>8d}ms  "
        f"{log['stages']['retrieve']['top_score']:>10.4f}  "
        f"{log['query'][:50]}"
    )

---

## 7. Log Analysis

Now we have real data. Let's load the logs back and compute summary statistics: average latency per stage, total token spend, and score distributions.

This is the bridge from "logging" to "observability" -- turning raw data into actionable insights.

In [ ]:
# Load all logs
logs = load_logs(log_dir=str(LOG_DIR))

# Extract metrics into lists for easy computation
embed_latencies = [log["stages"]["embed"]["latency_ms"] for log in logs]
retrieve_latencies = [log["stages"]["retrieve"]["latency_ms"] for log in logs]
generate_latencies = [log["stages"]["generate"]["latency_ms"] for log in logs]
total_latencies = [log["total_latency_ms"] for log in logs]

top_scores = [log["stages"]["retrieve"]["top_score"] for log in logs]
low_scores = [log["stages"]["retrieve"]["low_score"] for log in logs]

input_tokens = [log["stages"]["generate"]["input_tokens"] for log in logs]
output_tokens = [log["stages"]["generate"]["output_tokens"] for log in logs]

print("=== Latency Summary ===")
print(f"{'Stage':<12s}  {'Avg (ms)':>10s}  {'Min':>8s}  {'Max':>8s}  {'Std Dev':>10s}")
print("-" * 55)

for name, values in [("Embed", embed_latencies), ("Retrieve", retrieve_latencies),
                     ("Generate", generate_latencies), ("Total", total_latencies)]:
    avg = np.mean(values)
    print(f"{name:<12s}  {avg:>8.0f}ms  {min(values):>6d}ms  {max(values):>6d}ms  {np.std(values):>8.1f}ms")

print()
print("=== Retrieval Score Summary ===")
print(f"  Avg top-1 score:  {np.mean(top_scores):.4f}")
print(f"  Min top-1 score:  {min(top_scores):.4f}")
print(f"  Max top-1 score:  {max(top_scores):.4f}")
print(f"  Queries with top score < 0.5: {sum(1 for s in top_scores if s < 0.5)}/{len(top_scores)}")

print()
print("=== Token Usage Summary ===")
total_input = sum(input_tokens)
total_output = sum(output_tokens)
print(f"  Total input tokens:  {total_input:,}")
print(f"  Total output tokens: {total_output:,}")
print(f"  Avg input per query: {np.mean(input_tokens):,.0f}")
print(f"  Avg output per query: {np.mean(output_tokens):,.0f}")

In [ ]:
# Cost estimation
# Claude Sonnet 4.5 pricing (approximate):
#   Input:  $3.00 per million tokens
#   Output: $15.00 per million tokens
# Voyage AI embedding pricing (approximate):
#   ~$0.06 per million characters

SONNET_INPUT_COST_PER_M = 3.00
SONNET_OUTPUT_COST_PER_M = 15.00
VOYAGE_COST_PER_M_CHARS = 0.06

total_input = sum(input_tokens)
total_output = sum(output_tokens)
total_chars_embedded = sum(log["stages"]["embed"]["input_chars"] for log in logs)

gen_input_cost = (total_input / 1_000_000) * SONNET_INPUT_COST_PER_M
gen_output_cost = (total_output / 1_000_000) * SONNET_OUTPUT_COST_PER_M
embed_cost = (total_chars_embedded / 1_000_000) * VOYAGE_COST_PER_M_CHARS
total_cost = gen_input_cost + gen_output_cost + embed_cost

print("=== Cost Estimate ===")
print(f"  Queries:          {len(logs)}")
print(f"  Generation input: ${gen_input_cost:.6f}")
print(f"  Generation output: ${gen_output_cost:.6f}")
print(f"  Embedding:        ${embed_cost:.6f}")
print(f"  Total cost:       ${total_cost:.6f}")
print(f"  Cost per query:   ${total_cost / len(logs):.6f}")
print()
print(f"At this rate, 100 queries/day = ${total_cost / len(logs) * 100:.4f}/day")
print(f"               1000 queries/day = ${total_cost / len(logs) * 1000:.4f}/day")

---

## 8. Diagnostic Visualizations

Numbers tell you what happened. Charts show you the patterns. Let's build four diagnostic charts from our log data.

In [ ]:
# Chart 1: Latency breakdown by stage (stacked bar)
fig, ax = plt.subplots(figsize=(12, 5))

query_labels = [f"Q{i+1}" for i in range(len(logs))]
x = np.arange(len(logs))
width = 0.6

bars_embed = ax.bar(x, embed_latencies, width, label="Embed", color="#4A90D9")
bars_retrieve = ax.bar(x, retrieve_latencies, width, bottom=embed_latencies,
                       label="Retrieve", color="#50C878")
bars_generate = ax.bar(x, generate_latencies, width,
                       bottom=[e + r for e, r in zip(embed_latencies, retrieve_latencies)],
                       label="Generate", color="#FF6B6B")

ax.set_xlabel("Query")
ax.set_ylabel("Latency (ms)")
ax.set_title("Pipeline Latency Breakdown by Stage")
ax.set_xticks(x)
ax.set_xticklabels(query_labels)
ax.legend()

plt.tight_layout()
plt.show()

print("Generation dominates latency in nearly every query.")
print("This is expected -- Claude API calls are the most expensive operation.")

In [ ]:
# Chart 2: Similarity score distribution (histogram)
# Collect ALL retrieval scores across all queries
all_scores = []
for log in logs:
    # We stored top_score and low_score but not all individual scores
    # For a more complete view, let's use the top and low scores from each query
    all_scores.append(log["stages"]["retrieve"]["top_score"])
    all_scores.append(log["stages"]["retrieve"]["low_score"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top-1 scores across queries
axes[0].hist(top_scores, bins=10, color="#4A90D9", edgecolor="white", alpha=0.8)
axes[0].axvline(x=0.5, color="red", linestyle="--", label="Threshold (0.5)")
axes[0].set_xlabel("Similarity Score")
axes[0].set_ylabel("Number of Queries")
axes[0].set_title("Top-1 Retrieval Score Distribution")
axes[0].legend()

# Score spread per query
spreads = [log["stages"]["retrieve"]["score_spread"] for log in logs]
axes[1].bar(query_labels, spreads, color="#50C878", edgecolor="white")
axes[1].set_xlabel("Query")
axes[1].set_ylabel("Score Spread (top - bottom)")
axes[1].set_title("Retrieval Score Spread per Query")

plt.tight_layout()
plt.show()

print("Low top scores (< 0.5) mean retrieval is struggling to find relevant chunks.")
print("Narrow spread means all results are similarly (ir)relevant.")
print("Wide spread means one chunk is clearly better than the rest -- that's usually good.")

In [ ]:
# Chart 3: Token usage per query (bar chart)
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(logs))
width = 0.35

bars_in = ax.bar(x - width/2, input_tokens, width, label="Input Tokens", color="#4A90D9")
bars_out = ax.bar(x + width/2, output_tokens, width, label="Output Tokens", color="#FF6B6B")

ax.set_xlabel("Query")
ax.set_ylabel("Token Count")
ax.set_title("Token Usage per Query")
ax.set_xticks(x)
ax.set_xticklabels(query_labels)
ax.legend()

plt.tight_layout()
plt.show()

print(f"Avg input tokens: {np.mean(input_tokens):,.0f}")
print(f"Avg output tokens: {np.mean(output_tokens):,.0f}")
print()
print("Input tokens are driven by how many chunks you pass to Claude.")
print("If one query has much higher input tokens, check how many chunks were stuffed in.")

In [ ]:
# Chart 4: Cost accumulation over queries (line chart)
cumulative_costs = []
running_total = 0.0

for log in logs:
    gen = log["stages"]["generate"]
    emb = log["stages"]["embed"]

    query_cost = (
        (gen["input_tokens"] / 1_000_000) * SONNET_INPUT_COST_PER_M
        + (gen["output_tokens"] / 1_000_000) * SONNET_OUTPUT_COST_PER_M
        + (emb["input_chars"] / 1_000_000) * VOYAGE_COST_PER_M_CHARS
    )
    running_total += query_cost
    cumulative_costs.append(running_total)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(range(1, len(logs) + 1), cumulative_costs, marker="o", color="#4A90D9",
        linewidth=2, markersize=6)
ax.fill_between(range(1, len(logs) + 1), cumulative_costs, alpha=0.1, color="#4A90D9")

ax.set_xlabel("Number of Queries")
ax.set_ylabel("Cumulative Cost ($)")
ax.set_title("Cost Accumulation Over Queries")
ax.set_xticks(range(1, len(logs) + 1))

# Add cost labels
for i, cost in enumerate(cumulative_costs):
    ax.annotate(f"${cost:.5f}", (i + 1, cost), textcoords="offset points",
                xytext=(0, 10), ha="center", fontsize=8)

plt.tight_layout()
plt.show()

per_query = cumulative_costs[-1] / len(logs)
print(f"Total cost for {len(logs)} queries: ${cumulative_costs[-1]:.6f}")
print(f"Average cost per query: ${per_query:.6f}")
print(f"Projected daily cost (100 queries): ${per_query * 100:.4f}")
print(f"Projected daily cost (1000 queries): ${per_query * 1000:.4f}")

**What the charts tell you:**

1. **Latency breakdown** -- Generation dominates. If you need to optimize speed, focus there (fewer chunks, smaller context window, or a faster model).
2. **Score distribution** -- Queries with low top-1 scores are retrieval failures. These need investigation: is the query too vague? Is the answer missing from the corpus?
3. **Token usage** -- Consistent token counts mean your pipeline is predictable. Spikes mean one query got an unusually large context.
4. **Cost accumulation** -- Linear growth is predictable and budgetable. Non-linear would indicate some queries are much more expensive than others.

---

## 9. The Whodunit Challenge

Below are three pipeline log entries. Each one represents a **real failure mode**. Your job: read the log data, diagnose the problem, and write your analysis.

For each case, answer:
- **Which stage failed?** (embed, retrieve, or generate)
- **What is the root cause?**
- **What would you change to fix it?**

In [ ]:
# ============================================================
# CASE 1: The Slow Embed
# ============================================================

case_1 = {
    "query_id": "whodunit_01",
    "query": "What are the expense reimbursement procedures?",
    "timestamp": "2025-06-15T10:23:41",
    "stages": {
        "embed": {
            "latency_ms": 1847,
            "model": "voyage-3-large",
            "input_chars": 48
        },
        "retrieve": {
            "latency_ms": 45,
            "n_results": 5,
            "top_score": 0.31,
            "low_score": 0.18,
            "score_spread": 0.13,
            "sources": [
                "meeting_notes_q3.md",
                "employee_handbook.md",
                "standup_notes_jan.md",
                "security_update.md",
                "meeting_notes_q3.md"
            ],
            "unique_sources": 4,
            "filter_applied": None
        },
        "generate": {
            "latency_ms": 1523,
            "model": "claude-sonnet-4-5",
            "input_tokens": 2341,
            "output_tokens": 187,
            "stop_reason": "end_turn"
        }
    },
    "total_latency_ms": 3415
}

print("=== CASE 1: The Slow Embed ===")
print(json.dumps(case_1, indent=2))

### Your Diagnosis -- Case 1

**Which stage failed?**

*(Write your answer here)*

**What is the root cause?**

*(Write your answer here)*

**What would you change?**

*(Write your answer here)*

In [ ]:
# ============================================================
# CASE 2: The Vague Query
# ============================================================

case_2 = {
    "query_id": "whodunit_02",
    "query": "Tell me about the stuff",
    "timestamp": "2025-06-15T10:31:12",
    "stages": {
        "embed": {
            "latency_ms": 98,
            "model": "voyage-3-lite",
            "input_chars": 24
        },
        "retrieve": {
            "latency_ms": 22,
            "n_results": 5,
            "top_score": 0.28,
            "low_score": 0.21,
            "score_spread": 0.07,
            "sources": [
                "financial_report_q3.md",
                "employee_handbook.md",
                "meeting_notes_q3.md",
                "standup_notes_jan.md",
                "remote_work_policy.md"
            ],
            "unique_sources": 5,
            "filter_applied": None
        },
        "generate": {
            "latency_ms": 1102,
            "model": "claude-sonnet-4-5",
            "input_tokens": 2156,
            "output_tokens": 94,
            "stop_reason": "end_turn"
        }
    },
    "total_latency_ms": 1222
}

print("=== CASE 2: The Vague Query ===")
print(json.dumps(case_2, indent=2))

### Your Diagnosis -- Case 2

**Which stage failed?**

*(Write your answer here)*

**What is the root cause?**

*(Write your answer here)*

**What would you change?**

*(Write your answer here)*

In [ ]:
# ============================================================
# CASE 3: The Confidence Mirage
# ============================================================

case_3 = {
    "query_id": "whodunit_03",
    "query": "What is the maximum PTO carryover for senior employees?",
    "timestamp": "2025-06-15T10:45:33",
    "stages": {
        "embed": {
            "latency_ms": 105,
            "model": "voyage-3-lite",
            "input_chars": 55
        },
        "retrieve": {
            "latency_ms": 19,
            "n_results": 5,
            "top_score": 0.87,
            "low_score": 0.74,
            "score_spread": 0.13,
            "sources": [
                "vacation_policy_2025.md",
                "vacation_policy_2025.md",
                "employee_handbook.md",
                "vacation_policy_2025.md",
                "hr_policy.md"
            ],
            "unique_sources": 3,
            "filter_applied": None
        },
        "generate": {
            "latency_ms": 1876,
            "model": "claude-sonnet-4-5",
            "input_tokens": 2934,
            "output_tokens": 267,
            "stop_reason": "end_turn"
        }
    },
    "total_latency_ms": 2000,
    "_instructor_note": "The generated answer stated: 'Senior employees with 10+ years can carry over up to 10 days of PTO.' However, the retrieved chunks from vacation_policy_2025.md only discuss general PTO accrual rates and do not mention seniority-based carryover limits. The model hallucinated a specific number that does not appear in any source."
}

print("=== CASE 3: The Confidence Mirage ===")
# Print without the instructor note so students diagnose independently
display_case = {k: v for k, v in case_3.items() if k != "_instructor_note"}
print(json.dumps(display_case, indent=2))
print()
print("The generated answer was:")
print('  "Senior employees with 10+ years can carry over up to 10 days of PTO."')
print()
print("The retrieved chunks discuss general PTO accrual rates but do NOT mention")
print("seniority-based carryover limits anywhere.")

### Your Diagnosis -- Case 3

**Which stage failed?**

*(Write your answer here)*

**What is the root cause?**

*(Write your answer here)*

**What would you change?**

*(Write your answer here)*

### Whodunit Answer Key

*(Your instructor will discuss these after the exercise.)*

**Case 1 -- The Slow Embed:**
- Two problems. First, the embed stage took 1,847ms -- about 3x longer than normal. Look at the model: `voyage-3-large` instead of `voyage-3-lite`. Someone switched to the most expensive embedding model. Second, the retrieval scores are terrible (0.31 top score). This is likely because the query was embedded with a different model than the documents were embedded with. If your documents are embedded with `voyage-3-lite` and your query is embedded with `voyage-3-large`, the vectors are in different spaces. Retrieval fails.
- **Fix:** Use the same embedding model for queries and documents. Always.

**Case 2 -- The Vague Query:**
- Retrieval scores are uniformly low (0.28 top, 0.07 spread). The results come from 5 different documents -- nothing is a good match. The query "Tell me about the stuff" has no semantic specificity. The embedding cannot capture intent because there is no intent to capture. Notice the low output tokens (94) -- Claude correctly said it could not answer.
- **Fix:** This is a query quality problem, not a pipeline problem. In production, you might add query validation or suggest reformulation.

**Case 3 -- The Confidence Mirage:**
- High retrieval scores (0.87 top) -- retrieval worked. The right documents were found. But the answer contains a specific fact ("10+ years, 10 days carryover") that does not appear in any retrieved chunk. This is a **hallucination**: the model generated plausible-sounding content that goes beyond the provided context. The system prompt says "answer using ONLY the provided context," but the model violated that constraint.
- **Fix:** This is a generation/prompt problem. Strengthen the system prompt, reduce temperature, or add post-generation verification that checks whether specific claims appear in the source chunks.

---

## 10. Pipeline Health Dashboard

Let's build a reusable function that takes a list of log entries and prints a health summary. This is the kind of diagnostic view you would run after a batch of queries to check if your pipeline is healthy.

In [ ]:
def pipeline_health_report(logs: list[dict]) -> None:
    """Print a health summary from a list of pipeline log entries.

    Computes and displays:
    - Total queries processed
    - Average and p95 latency
    - Total token usage and estimated cost
    - Retrieval score statistics
    - The lowest-scoring query (most likely retrieval failure)
    """
    if not logs:
        print("No logs to analyze.")
        return

    # Latency
    total_latencies = [log["total_latency_ms"] for log in logs]
    avg_latency = np.mean(total_latencies)
    p95_latency = np.percentile(total_latencies, 95)

    # Tokens
    total_input = sum(log["stages"]["generate"]["input_tokens"] for log in logs)
    total_output = sum(log["stages"]["generate"]["output_tokens"] for log in logs)

    # Cost
    total_cost = (
        (total_input / 1_000_000) * SONNET_INPUT_COST_PER_M
        + (total_output / 1_000_000) * SONNET_OUTPUT_COST_PER_M
        + sum(log["stages"]["embed"]["input_chars"] for log in logs) / 1_000_000 * VOYAGE_COST_PER_M_CHARS
    )

    # Retrieval scores
    top_scores = [log["stages"]["retrieve"]["top_score"] for log in logs]
    avg_top_score = np.mean(top_scores)
    low_score_queries = sum(1 for s in top_scores if s < 0.5)

    # Find worst query
    worst_idx = np.argmin(top_scores)
    worst_log = logs[worst_idx]

    # Print report
    print("=" * 60)
    print("         PIPELINE HEALTH REPORT")
    print("=" * 60)
    print()
    print(f"  Total queries:        {len(logs)}")
    print(f"  Avg latency:          {avg_latency:,.0f}ms")
    print(f"  p95 latency:          {p95_latency:,.0f}ms")
    print()
    print(f"  Total input tokens:   {total_input:,}")
    print(f"  Total output tokens:  {total_output:,}")
    print(f"  Total estimated cost: ${total_cost:.6f}")
    print()
    print(f"  Avg top-1 score:      {avg_top_score:.4f}")
    print(f"  Low-score queries:    {low_score_queries}/{len(logs)} (top score < 0.5)")
    print()
    print(f"  Lowest-scoring query:")
    print(f"    Query:    {worst_log['query']}")
    print(f"    Top score: {worst_log['stages']['retrieve']['top_score']:.4f}")
    print(f"    Sources:  {worst_log['stages']['retrieve']['sources']}")
    print()
    print("=" * 60)


print("pipeline_health_report() defined.")

In [ ]:
# Run the health report on our logged queries
logs = load_logs(log_dir=str(LOG_DIR))
pipeline_health_report(logs)

In [ ]:
# Source frequency analysis -- which documents get retrieved most often?
from collections import Counter

all_sources = []
for log in logs:
    all_sources.extend(log["stages"]["retrieve"]["sources"])

source_counts = Counter(all_sources)

print("=== Source Retrieval Frequency ===")
print(f"{'Source':<40s}  {'Count':>6s}  {'Pct':>6s}")
print("-" * 55)

for source, count in source_counts.most_common():
    pct = count / len(all_sources) * 100
    print(f"{source:<40s}  {count:>6d}  {pct:>5.1f}%")

print()
print(f"Total retrievals: {len(all_sources)}")
print(f"Unique sources: {len(source_counts)}")
print()
print("If one source dominates, your corpus may be unbalanced.")
print("If a source never appears, its chunks may be poorly embedded.")

---

## 11. Bridge to Assessment

Session 4.2 is the **Module Test**. You will:

1. **Submit Lab 2** at the start of class (naive vs filtered RAG comparison)
2. **Demo your CLI** -- live queries with observability output
3. **Take the written exam** -- 50 minutes, closed-note

### How today's work helps

- **Lab 2:** If you wire `PipelineLogger` into both pipelines, your metrics capture is automated. The rubric explicitly evaluates: "Latency, similarity scores, retrieval quality logged and reported; clear patterns identified."
- **CLI Demo:** The structured logs give you something concrete to show. Run a query, show the trace, explain what each number means.
- **Written Exam:** 50% of the exam is debugging scenarios -- exactly like the Whodunit challenge. "Here are the logs, here are the scores -- what went wrong?"

### What to study

| Topic | Key Concepts | Session |
|-------|-------------|--------|
| API & Auth | Secret management, rate limits, error handling, token billing | 1.1 |
| Extraction | Pydantic schemas, structured output, batch processing | 1.2 |
| Embeddings | Dimensions, model tradeoffs, semantic similarity | 2.1 |
| Chunking | Fixed vs semantic vs overlap, metadata, size tradeoffs | 2.2 |
| Naive RAG | Retrieval mechanics, failure modes, similarity scoring | 3.1 |
| Filtered RAG | Metadata filters, when filtering helps vs hurts | 3.2 |
| Observability | Structured logging, metrics, retrieval autopsy | 4.1 |

**Best preparation:** Review your own pipeline. Can you explain every design decision? Can you trace a query through every stage? If you can do that, you will do well.

---

## 12. Session Recap

### What we built today

1. **Explored StageTimer** -- a context manager that captures elapsed time in milliseconds for any operation
2. **Explored PipelineLogger** -- structured logging for the full RAG pipeline: embed, retrieve, generate
3. **Wired logging into the pipeline** -- every query now produces a JSONL log entry with latency, scores, sources, and tokens
4. **Ran multiple queries** -- built up a log file with 10 diverse queries for analysis
5. **Analyzed logs** -- computed averages, distributions, and cost estimates from real pipeline data
6. **Built diagnostic visualizations** -- latency breakdowns, score distributions, token usage, cost accumulation
7. **Diagnosed failures** -- the Whodunit challenge: model mismatch, vague query, and hallucination
8. **Built a health dashboard** -- a reusable function that surfaces pipeline health indicators

### Key takeaways

- **Structured logging is not optional.** In non-deterministic systems, you must log what happened because you cannot reason about it after the fact.
- **Most RAG failures are retrieval failures.** If the right chunks do not reach the model, no prompt engineering saves you.
- **The retrieval autopsy is a systematic workflow.** Bad answer? Check the sources. Check the scores. Check the chunks. Check the embedding. Work backward.
- **Observability is the difference between debugging and guessing.** Without logs, you are guessing. With logs, you are engineering.
- **Cost is a metric too.** Every query has a price. Logging it lets you budget, optimize, and explain.

### What you have built across AI-2

| Session | What We Built | Status |
|---------|--------------|--------|
| 1.1 | API connection + generation | Done |
| 1.2 | Structured extraction with Pydantic | Done |
| 2.1 | Embeddings + similarity measurement | Done |
| 2.2 | Chunking + vector store ingestion | Done |
| 3.1 | Naive RAG pipeline | Done |
| 3.2 | Metadata-filtered RAG + evaluation | Done |
| **4.1** | **Observability + debugging** | **Done (today)** |
| 4.2 | Module Test | Next session |

### Questions to leave with

- **On Integration:** What is the hardest part of turning a collection of scripts into a working tool?
- **On Reflection:** What would you do differently if you built this pipeline again from scratch?
- **On Next Steps:** What can this CLI not do that a real application would need? *(Preview: AI-3)*